# Cost Inputs Exploration

Compare the technology costs currently used in `pypsa-earth/data/costs.csv` (mostly DIW 2010 / IEA 2010–2011 vintage, in **2013 EUR**) against DEA 2030 projections from our archived `tech_config_ammonia_plant_2030_dea.yaml` (in **2020 EUR**).

**TODO:** Decide which cost inputs to use for final runs. Options:
1. Keep current costs.csv as-is (legacy PyPSA-Earth defaults, 2013 EUR)
2. Update costs.csv to DEA 2030 values (converted to 2013 EUR for consistency)
3. Update costs.csv to DEA 2030 values in 2020 EUR and adjust the model's currency year

Key question: do we want internal consistency with upstream PyPSA-Earth, or accuracy relative to our DEA reference case?

In [1]:
import pandas as pd
import yaml
from pathlib import Path
from IPython.display import display, HTML

## Load current model costs (costs.csv)

In [2]:
costs_csv = pd.read_csv("../pypsa-earth/data/costs.csv")

# Pivot to get investment, FOM, VOM, lifetime, efficiency, fuel per technology
costs_wide = costs_csv.pivot_table(
    index="technology", columns="parameter", values="value", aggfunc="first"
)

# Focus on the key technologies we want to compare
key_techs = [
    "solar", "onwind", "offwind", "nuclear", "OCGT", "CCGT",
    "electrolysis", "fuel cell", "hydrogen storage tank",
    "battery storage", "battery inverter",
    "ammonia synthesis", "ammonia storage",
    "CCGT H2", "CCGT NH3", "NH3 pipeline", "H2 pipeline",
]
available_techs = [t for t in key_techs if t in costs_wide.index]

cols_of_interest = ["investment", "FOM", "VOM", "lifetime", "efficiency", "fuel"]
cols_available = [c for c in cols_of_interest if c in costs_wide.columns]

current_costs = costs_wide.loc[available_techs, cols_available].copy()

# Get sources/currency year from raw CSV
inv_rows = costs_csv[
    (costs_csv["technology"].isin(available_techs)) & (costs_csv["parameter"] == "investment")
][["technology", "source", "currency_year"]].set_index("technology")

current_costs["source"] = inv_rows["source"]
current_costs["currency_year"] = inv_rows["currency_year"]

print("=== Current Model Costs (costs.csv) ===")
print("Note: investment is in EUR/kW (generators/links) or EUR/kWh (stores)")
print("All monetary values are in the currency_year shown (mostly 2013 EUR)\n")
display(current_costs.style.format(precision=2, na_rep="—"))

=== Current Model Costs (costs.csv) ===
Note: investment is in EUR/kW (generators/links) or EUR/kWh (stores)
All monetary values are in the currency_year shown (mostly 2013 EUR)



parameter,investment,FOM,VOM,lifetime,efficiency,fuel,source,currency_year
technology,,,,,,,,
solar,600.00,4.17,0.01,25.00,—,—,DIW DataDoc http://hdl.handle.net/10419/80348,2013.00
onwind,1040.00,2.45,2.30,30.00,—,—,DEA https://ens.dk/en/our-services/projections-and-models/technology-data,2013.00
offwind,1640.00,2.30,2.70,30.00,—,—,DEA https://ens.dk/en/our-services/projections-and-models/technology-data,2013.00
nuclear,6000.00,—,8.00,45.00,0.34,3.00,DIW DataDoc http://hdl.handle.net/10419/80348,2013.00
OCGT,400.00,3.75,3.00,30.00,0.39,—,DIW DataDoc http://hdl.handle.net/10419/80348,2013.00
CCGT,800.00,2.50,4.00,30.00,0.50,—,DIW DataDoc http://hdl.handle.net/10419/80348,2013.00
electrolysis,669.34,2.00,—,25.00,0.74,—,DEA 2030 AEC 1 GW,—
fuel cell,339.00,3.00,—,20.00,0.58,—,NREL http://www.nrel.gov/docs/fy09osti/45873.pdf,2013.00
hydrogen storage tank,11.20,—,—,20.00,—,—,budischak2013,2013.00


## Load DEA 2030 costs (archived tech config)

In [ ]:
with open("../archive/tech_config_ammonia_plant_2030_dea.yaml") as f:
    dea_config = yaml.safe_load(f)

# Extract DEA costs into a comparable DataFrame
dea_rows = []
for tech_name, tech_data in dea_config["techs"].items():
    ctype = tech_data.get("component_type", "")
    if ctype == "generator" or ctype == "link":
        cost_key = "overnight_cost_per_mw"
        unit = "EUR/kW"
        divisor = 1000  # Convert EUR/MW to EUR/kW
    elif ctype == "store":
        cost_key = "overnight_cost_per_mwh"
        unit = "EUR/kWh"
        divisor = 1000
    else:
        continue
    
    # Skip penalty/placeholder components
    if "penalty" in tech_name:
        continue
    
    inv = tech_data.get(cost_key, 0) / divisor
    dea_rows.append({
        "dea_tech": tech_name,
        "component_type": ctype,
        "investment_2020EUR": inv,
        "unit": unit,
        "lifetime": tech_data.get("lifetime_years"),
        "FOM_pct": tech_data.get("fixed_om_fraction", 0) * 100,
        "efficiency": tech_data.get("overall_efficiency"),
        "source": tech_data.get("source_note", ""),
    })

# Add DEA 2030 technologies not in the archived YAML
# CCGT: DEA "Gas turb. CC, steam extract." — 882.6 EUR/kW, eff 0.61, lifetime 25y, FOM 3.4%
dea_rows.append({
    "dea_tech": "ccgt_gas",

    "component_type": "generator",display(dea_costs.style.format(precision=2, na_rep="—"))

    "investment_2020EUR": 882.6,  # EUR/kW (already in kW basis)print("=== DEA 2030 Costs (from archived tech config + manual entries, 2020 EUR) ===")

    "unit": "EUR/kW",

    "lifetime": 25,dea_costs = pd.DataFrame(dea_rows).set_index("dea_tech")

    "FOM_pct": 3.4,

    "efficiency": 0.61,})
    "source": "DEA 2030 Gas turb. CC, steam extract.",

=== DEA 2030 Costs (from archived tech config, 2020 EUR) ===


,component_type,investment_2020EUR,unit,lifetime,FOM_pct,efficiency,source
dea_tech,,,,,,,
solar,generator,380.00,EUR/kW,40,2.00,—,DEA 2030 Utility-scale PV
solar_tracking,generator,450.00,EUR/kW,40,2.00,—,DEA 2030 Utility-scale PV tracker
onshore_wind,generator,930.00,EUR/kW,30,2.00,—,DEA 2030 Onshore turbines
offshore_wind_fixed,generator,2393.13,EUR/kW,30,2.00,—,"DEA 2030 Offshore Wind AC Fixed, max depth 50m (component regrouping from DEA line items)"
offshore_wind_floating,generator,3558.13,EUR/kW,30,2.00,—,"DEA 2030 Offshore Wind AC Floating, max depth 1000m (component regrouping from DEA line items)"
ammonia_synthesis,link,1526.00,EUR/kW,30,3.00,0.77,DEA 2030 Hydrogen to Ammonia
electrolysis,link,669.34,EUR/kW,25,2.00,0.74,2030 DEA AEC 1 GW
hydrogen_compression,link,495.86,EUR/kW,20,2.00,0.95,Computed from 2030 DEA hydrogen storage tanks sheet (component-share regrouping)
hydrogen_fuel_cell,link,2475.00,EUR/kW,20,2.00,0.50,fuelCell (no direct DEA line in the imported sheets; converted from Queensland report)


## Side-by-side comparison: Current vs DEA 2030

Map DEA tech names to costs.csv tech names for direct comparison.

To deflate DEA 2020 EUR → 2013 EUR, we use Eurozone HICP cumulative inflation of ~8.4% over 2013–2020 (ECB data), i.e. deflator = 1/1.084 ≈ 0.9225.

In [ ]:
# Mapping: DEA tech name → costs.csv tech name
dea_to_csv_map = {
    "solar": "solar",
    "onshore_wind": "onwind",
    "offshore_wind_fixed": "offwind",
    "ccgt_gas": "CCGT",
    # No DEA nuclear — DIW only
    "electrolysis": "electrolysis",
    "hydrogen_fuel_cell": "fuel cell",
    "ammonia_synthesis": "ammonia synthesis",
    "battery_pcs_charge": "battery inverter",
    "battery_storage": "battery storage",
    "compressed_hydrogen_store": "hydrogen storage tank",
    "ammonia": "ammonia storage",
}

# Eurozone HICP cumulative inflation 2013→2020: ~8.4%
# Source: ECB Statistical Data Warehouse, HICP overall index
DEFLATOR_2020_TO_2013 = 1 / 1.084  # ≈ 0.9225

comparison_rows = []
for dea_name, csv_name in dea_to_csv_map.items():
    if dea_name not in dea_costs.index:
        continue
    dea_row = dea_costs.loc[dea_name]
    
    csv_inv = current_costs.loc[csv_name, "investment"] if csv_name in current_costs.index else None
    csv_lifetime = current_costs.loc[csv_name, "lifetime"] if csv_name in current_costs.index else None
    csv_source = current_costs.loc[csv_name, "source"] if csv_name in current_costs.index else "—"
    
    dea_inv_2020 = dea_row["investment_2020EUR"]
    dea_inv_2013 = dea_inv_2020 * DEFLATOR_2020_TO_2013
    
    pct_diff_vs_2020 = (
        (csv_inv - dea_inv_2020) / dea_inv_2020 * 100 if csv_inv and dea_inv_2020 else None
    )
    pct_diff_vs_2013 = (
        (csv_inv - dea_inv_2013) / dea_inv_2013 * 100 if csv_inv and dea_inv_2013 else None
    )
    
    comparison_rows.append({
        "technology": csv_name,
        "dea_tech": dea_name,
        "unit": dea_row["unit"],
        "current_csv (2013 EUR)": csv_inv,
        "DEA 2030 (2020 EUR)": dea_inv_2020,
        "DEA 2030 (→2013 EUR)": dea_inv_2013,
        "csv vs DEA_2020 (%)": pct_diff_vs_2020,
        "csv vs DEA_2013 (%)": pct_diff_vs_2013,
        "csv_source": csv_source,
    })

comp_df = pd.DataFrame(comparison_rows).set_index("technology")

print("=== Investment Cost Comparison: Current costs.csv vs DEA 2030 ===")
print("Positive % = costs.csv is MORE expensive than DEA")
print("Negative % = costs.csv is CHEAPER than DEA\n")
display(
    comp_df.style.format({
        "current_csv (2013 EUR)": "{:,.1f}",
        "DEA 2030 (2020 EUR)": "{:,.1f}",
        "DEA 2030 (→2013 EUR)": "{:,.1f}",
        "csv vs DEA_2020 (%)": "{:+.1f}%",
        "csv vs DEA_2013 (%)": "{:+.1f}%",
    }, na_rep="—")
    .map(
        lambda v: "color: green" if isinstance(v, str) and v.startswith("+") else
                  ("color: red" if isinstance(v, str) and v.startswith("-") else ""),
        subset=["csv vs DEA_2020 (%)", "csv vs DEA_2013 (%)"]
    )
)

=== Investment Cost Comparison: Current costs.csv vs DEA 2030 ===
Positive % = costs.csv is MORE expensive than DEA
Negative % = costs.csv is CHEAPER than DEA



,dea_tech,unit,current_csv (2013 EUR),DEA 2030 (2020 EUR),DEA 2030 (→2013 EUR),csv vs DEA_2020 (%),csv vs DEA_2013 (%),csv_source
technology,,,,,,,,
solar,solar,EUR/kW,600.0,380.0,350.6,+57.9%,+71.2%,DIW DataDoc http://hdl.handle.net/10419/80348
onwind,onshore_wind,EUR/kW,"1,040.0",930.0,857.9,+11.8%,+21.2%,DEA https://ens.dk/en/our-services/projections-and-models/technology-data
offwind,offshore_wind_fixed,EUR/kW,"1,640.0","2,393.1","2,207.7",-31.5%,-25.7%,DEA https://ens.dk/en/our-services/projections-and-models/technology-data
electrolysis,electrolysis,EUR/kW,669.3,669.3,617.5,+0.0%,+8.4%,DEA 2030 AEC 1 GW
fuel cell,hydrogen_fuel_cell,EUR/kW,339.0,"2,475.0","2,283.2",-86.3%,-85.2%,NREL http://www.nrel.gov/docs/fy09osti/45873.pdf
ammonia synthesis,ammonia_synthesis,EUR/kW,"1,526.0","1,526.0","1,407.7",+0.0%,+8.4%,DEA 2030 Hydrogen to Ammonia
battery inverter,battery_pcs_charge,EUR/kW,411.0,149.0,137.5,+175.8%,+199.0%,budischak2013
battery storage,battery_storage,EUR/kWh,192.0,130.0,119.9,+47.7%,+60.1%,budischak2013
hydrogen storage tank,compressed_hydrogen_store,EUR/kWh,11.2,25.0,23.1,-55.2%,-51.4%,budischak2013


## Lifetime comparison

In [5]:
lifetime_rows = []
for dea_name, csv_name in dea_to_csv_map.items():
    if dea_name not in dea_costs.index:
        continue
    dea_lt = dea_costs.loc[dea_name, "lifetime"]
    csv_lt = current_costs.loc[csv_name, "lifetime"] if csv_name in current_costs.index else None
    
    lifetime_rows.append({
        "technology": csv_name,
        "csv_lifetime_yr": csv_lt,
        "dea_lifetime_yr": dea_lt,
        "difference_yr": (csv_lt - dea_lt) if csv_lt and dea_lt else None,
    })

lt_df = pd.DataFrame(lifetime_rows).set_index("technology")
print("=== Lifetime Comparison ===")
display(lt_df)

=== Lifetime Comparison ===


,csv_lifetime_yr,dea_lifetime_yr,difference_yr
technology,,,
solar,25.0,40,-15.0
onwind,30.0,30,0.0
offwind,30.0,30,0.0
electrolysis,25.0,25,0.0
fuel cell,20.0,20,0.0
ammonia synthesis,30.0,30,0.0
battery inverter,20.0,20,0.0
battery storage,15.0,20,-5.0
hydrogen storage tank,20.0,30,-10.0


## Technologies without DEA equivalents

These technologies in our costs.csv have no DEA 2030 counterpart in the archived config:

In [6]:
techs_with_dea = set(dea_to_csv_map.values())
techs_without_dea = [t for t in available_techs if t not in techs_with_dea]

if techs_without_dea:
    no_dea = current_costs.loc[techs_without_dea, ["investment", "lifetime", "source", "currency_year"]]
    print("=== Technologies in costs.csv with NO DEA 2030 equivalent ===")
    display(no_dea)
else:
    print("All key technologies have DEA equivalents.")

=== Technologies in costs.csv with NO DEA 2030 equivalent ===


parameter,investment,lifetime,source,currency_year
technology,,,,
nuclear,6000.0,45.0,DIW DataDoc http://hdl.handle.net/10419/80348,2013.0
OCGT,400.0,30.0,DIW DataDoc http://hdl.handle.net/10419/80348,2013.0
CCGT,800.0,30.0,DIW DataDoc http://hdl.handle.net/10419/80348,2013.0
CCGT H2,800.0,30.0,user,NaN
CCGT NH3,800.0,30.0,user,NaN
NH3 pipeline,267.0,40.0,user,NaN
H2 pipeline,267.0,40.0,Welder et al https://doi.org/10.1016/j.ijhyden...,2013.0


## Summary & TODO

**Key observations:**
- Solar and wind costs in costs.csv are from DIW 2010 / DEA (unclear vintage), in 2013 EUR
- Nuclear costs (6,000 EUR/kW) are from DIW 2010, well below modern European new-build (~7,400–10,500 EUR/kW)
- Electrolysis and ammonia costs were already sourced from DEA 2030
- Battery storage costs may differ significantly between DIW and DEA
- Fuel cell costs (339 EUR/kW from NREL 2009) are vastly cheaper than DEA's 2,475 EUR/kW — but fuel cells are currently disabled in our model (replaced by CCGT H2)

**TODO for final runs:**
1. Decide whether to update all costs.csv entries to DEA 2030 (deflated to 2013 EUR)
2. If updating, verify that the DEA offshore wind cost (~2,393 EUR/kW) is appropriate for AC-connected (our model uses offwind-ac and offwind-dc separately)
3. Nuclear: consider using DEA 2030 or more recent estimates (~7,400 EUR/kW) instead of DIW's 6,000 EUR/kW
4. Note that changing costs will invalidate all existing results — a full re-run would be needed
5. Consider whether 2013 EUR or 2020 EUR is the better base for reporting